# Reproducing *Cross-Dataset Robustness of Spiral Classifiers for Parkinson's Disease*

This single notebook reproduces every experiment in the paper, in the paper's naming.
It has three sections:

| Section | What | Data? | GPU? | Time |
|---|---|---|---|---|
| **0. Regenerate tables & figures** | Rebuild all paper numbers from a local `results/` directory | No | No | seconds |
| **1. Re-train (CPU pipelines)** | Re-run DINOv2 + Folador HOG from scratch | Yes | No | minutes–hours |
| **2. Re-train (GPU pipelines)** | Re-run Huang VGG19 + Dogan triple from scratch | Yes | Yes | hours |

**Most readers only need Section 0.** It is the headline reproducibility claim: no
datasets, no GPU, no training — every table and figure is recomputed from the
curated 97-cell result set in `results/`.

> **`results/` is not distributed with this repository** — it is git-ignored, and the
> parquet files are produced locally. Obtain them from the authors, or generate them
> yourself with Sections 1 and 2, before running Section 0.

Sections 1 and 2 re-train from raw images and require the datasets on disk (see
`docs/DATA.md`) plus a built manifest. They are idempotent: a cell whose
predictions already exist is skipped.

**Dataset names.** The paper says *Zham*, *Folador* and *Moura*; the code keys them
`kaggle`, `folador` and `hemato`. The keys are frozen and appear verbatim in every
command and result file — read `hemato` as **Moura**.

> Run from the repo root after `pip install -e .` (Section 0) or
> `pip install -e ".[train]"` (Sections 1–2).

In [ ]:
# --- setup: make the package importable and locate the shipped results ---
import sys
from pathlib import Path

# Works whether the notebook is run from notebooks/ or the repo root.
REPO = Path.cwd()
if (REPO / "reproduce_experiments.ipynb").exists():
    REPO = REPO.parent
assert (REPO / "src" / "parkinson_spiral").is_dir(), f"run from the repo (got {REPO})"
sys.path.insert(0, str(REPO / "src"))

RESULTS = REPO / "results"
ARTIFACTS = REPO / "paper_artifacts"
print("repo:    ", REPO)
print("results: ", RESULTS)

## 0. Regenerate the paper's tables and figures (no data, no GPU)

Everything below reads only your local `results/` directory and recomputes
the paper's numbers from scratch.

In [ ]:
from parkinson_spiral.analysis import cross_matrix, load_results, offdiagonal_mean, subject_suite
from parkinson_spiral.catalog import EXPECTED_N_CELLS, all_cells

cells, preds = load_results(RESULTS)
print(f"shipped cells:       {len(cells)}")
print(f"shipped predictions: {len(preds):,} rows across {preds['cell_id'].nunique()} cells")

# Catalog is the declarative source of truth: it must enumerate exactly these cells.
catalog_ids = {c.cell_id for c in all_cells()}
shipped_ids = set(cells["cell_id"])
assert len(catalog_ids) == EXPECTED_N_CELLS == 97
assert catalog_ids == shipped_ids, "catalog and shipped results disagree"
print(f"\ncatalog matches shipped results exactly: {EXPECTED_N_CELLS} cells ✓")

In [ ]:
# Regenerate the four paper tables + the two TikZ figures into paper_artifacts/.
import subprocess

from IPython.display import Markdown, display

proc = subprocess.run(
    [
        sys.executable,
        str(REPO / "scripts" / "paper.py"),
        "regenerate",
        "--results",
        str(RESULTS),
        "--out",
        str(ARTIFACTS),
    ],
    cwd=REPO,
    capture_output=True,
    text=True,
)
print(proc.stdout or proc.stderr)
proc.check_returncode()

display(Markdown((ARTIFACTS / "results_tables.md").read_text(encoding="utf-8")))

### The cross-dataset matrices (H2), computed live

Each pipeline's 3×3 train × test subject-accuracy matrix. The **diagonal** is the
in-domain CV ceiling; the **off-diagonal** is cross-dataset transfer. The paper's
H2 finding is that off-diagonal accuracy collapses toward chance — the classifiers
do not transfer across datasets.

In [ ]:
PIPELINES = [
    ("dinov2_probe", "AUG-None"),
    ("huang_vgg19", "AUG-Huang"),
    ("dogan_triple", "AUG-Dogan"),
    ("folador_hog", "AUG-None"),
]
for pipeline, recipe in PIPELINES:
    m = cross_matrix(cells, preds, pipeline, recipe)
    off = offdiagonal_mean(cells, preds, pipeline, recipe)
    print(f"\n{pipeline}  ({recipe})   off-diagonal mean acc = {off:.3f}")
    print(m.to_string(index=False))

In [ ]:
# The DINOv2 in-domain CV ceiling per dataset, full metric suite + bootstrap 95% CI.
for source in ("kaggle", "folador", "hemato"):
    s = subject_suite(
        cells,
        preds,
        pipeline_name="dinov2_probe",
        recipe_name="AUG-None",
        train_source=source,
        test_source=source,
        split_source="cv-fold",
    )
    print(
        f"DINOv2 (AUG-None) on {source:8s}  "
        f"acc={s['accuracy']:.3f} [{s['ci_lo']:.3f}–{s['ci_hi']:.3f}]  "
        f"auc={s['auc']:.3f}  n_subjects={s['n_subjects']}"
    )

## 1. Re-train from scratch — CPU pipelines

Re-running the experiments reproduces `results/predictions/` from the raw images.
**Prerequisites** (see `docs/DATA.md` and `docs/REPRODUCE.md`):

1. `pip install -e ".[train]"`
2. Datasets placed under a `<data-root>` (Zham + Folador are public; Moura, keyed `hemato`, on request).
3. A built manifest: `python scripts/paper.py build-manifest --data-root <path> --out results/manifest.parquet`

The pipelines below run comfortably on **CPU**:

- `dinov2_probe` — frozen public backbone → features → one logistic fit.
- `folador_hog` — deterministic HOG features → SVM head.

The next cell only runs if a manifest exists; otherwise it prints how to build one
and skips (so the notebook stays runnable end-to-end without data).

In [ ]:
MANIFEST = RESULTS / "manifest.parquet"
OUT = REPO / "runs" / "notebook"
CPU_PIPELINES = ["dinov2_probe", "folador_hog"]


def reproduce(pipelines):
    cmd = [
        sys.executable,
        str(REPO / "scripts" / "paper.py"),
        "reproduce",
        "--manifest",
        str(MANIFEST),
        "--out",
        str(OUT),
    ]
    for p in pipelines:
        cmd += ["--pipeline", p]
    proc = subprocess.run(cmd, cwd=REPO, capture_output=True, text=True)
    print(proc.stdout[-4000:] or proc.stderr)
    proc.check_returncode()


if not MANIFEST.is_file():
    print("No manifest found at", MANIFEST)
    print("Build one first (needs the datasets on disk):")
    print("  python scripts/paper.py build-manifest --data-root <path> --out results/manifest.parquet")
    print("\nSkipping re-training (Section 0 already reproduced every number).")
else:
    reproduce(CPU_PIPELINES)
    print("\nVerify the fresh run matches the catalog:")
    print(f"  python scripts/paper.py verify-catalog --results {OUT}")

## 2. Re-train from scratch — GPU pipelines

These pipelines need a **GPU** in practice:

- `huang_vgg19` — VGG19 fine-tuned end-to-end, 50 epochs.
- `dogan_triple` — three frozen CNN backbones + NCA + linear SVM.

The cell below guards on `torch.cuda.is_available()` and skips (rather than
running for many hours on CPU) when no GPU is present. The Dogan **Moura CV**
folds are small enough to run on CPU in Section 1-style if you really need them,
but the full matrices are GPU work.

`huang_vgg19` deliberately aborts with a non-zero exit if the fine-tune collapses
to a single-class predictor on its own training data, or produces no usable
gradient. That is a guard, not a crash — see `docs/REPRODUCE.md`.

In [ ]:
GPU_PIPELINES = ["huang_vgg19", "dogan_triple"]

try:
    import torch

    has_gpu = torch.cuda.is_available()
except ImportError:
    has_gpu = False
    print('torch not installed — install the training extra: pip install -e ".[train]"')

if not has_gpu:
    print("No CUDA GPU detected — skipping the GPU pipelines (they are impractical on CPU).")
    print("GPU pipelines:", GPU_PIPELINES)
elif not MANIFEST.is_file():
    print("No manifest — see Section 1 for how to build one. Skipping.")
else:
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    reproduce(GPU_PIPELINES)
    print("\nVerify:  python scripts/paper.py verify-catalog")

---

Done. Section 0 reproduces every paper number from a local `results/` directory; Sections 1–2
rebuild those results from raw images. For the released DINOv2 + Moura probe, see
`models/MODEL_CARD.md` and `models/predict.py`. Provenance and the one documented
AUC nuance are in `docs/PROVENANCE.md`.